In [2]:
import os, json, gc, glob
import numpy as np, pandas as pd, torch
import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

FOLDER    = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
FEATURES  = os.path.join(FOLDER, "zonal_features.parquet")
LEADS     = os.path.join(FOLDER, "weather_leads_zonal.parquet")
CKPT      = os.path.join(FOLDER, "checkpoints_zonal", "zonal_tft_lr3e4_best.ckpt")
BASE_JSON = os.path.join(FOLDER, "metrics_lead_matched.json")
OUT_JSON  = os.path.join(FOLDER, "metrics_zonal_lr3e4.json")

for f in glob.glob(os.path.join(FOLDER, "zonal_lead*.npz")):
    os.remove(f); print("removed", os.path.basename(f), flush=True)

TRAIN_START="2015-07-01 04:00"; VAL_START="2023-01-01 05:00"; TEST_START="2024-01-23 12:00"
ENCODER_LEN, DECODER_LEN = 168, 120
BATCH, SEED = 128, 42

ZONE_WEATHER = {"WEST":"A_WEST","GENESE":"B_GENESE","CENTRL":"C_CENTRL","NORTH":"D_NORTH",
    "MHK VL":"E_MHKVL","CAPITL":"F_CAPITL","HUD VL":"G_HUDVL","MILLWD":"G_HUDVL",
    "DUNWOD":"J_NYC","N.Y.C.":"J_NYC","LONGIL":"K_LONGIL"}
LEAN_VARS=["temperature_2m","apparent_temperature","relative_humidity_2m","wind_speed_10m","shortwave_radiation","cloud_cover"]
UNKNOWN_REALS=["demand","demand_lag24","demand_lag168","demand_roll24_mean","demand_roll168_mean","demand_roll24_std"]
KNOWN_REALS=LEAN_VARS+["temp_vshape","time_idx"]
KNOWN_CATS=["hour","day_of_week","month","is_weekend","is_holiday"]

def lead_file(d): return os.path.join(FOLDER, f"zonal_lead{d}.npz")

def load_base():
    df=pd.read_parquet(FEATURES); df["utc"]=pd.to_datetime(df["utc"])
    for c in KNOWN_CATS: df[c]=df[c].astype(str).astype("category")
    df["zone"]=df["zone"].astype(str)
    df=df[df["utc"]>=TRAIN_START].copy()
    df["time_idx"]=df["time_idx"]-df["time_idx"].min()
    return df.sort_values(["zone","time_idx"]).reset_index(drop=True)

def swap_lead(base, leads, d):
    out=base.copy(); boundary=pd.Timestamp(TEST_START); fb=0
    for zone,wkey in ZONE_WEATHER.items():
        mask=out["zone"]==zone; utc=out.loc[mask,"utc"]; post=utc>=boundary
        for v in LEAN_VARS:
            s=utc.map(leads[f"{v}_prev_day{d}__{wkey}"]).interpolate(limit=6)
            s5=utc.map(leads[f"{v}_prev_day5__{wkey}"]).interpolate(limit=6)
            s=s.fillna(s5); fb+=int((s.isna()&post).sum())
            s=s.fillna(out.loc[mask,v]); out.loc[mask,v]=s.values
    out["temp_vshape"]=(out["temperature_2m"]-14.0).abs()
    print(f"  fallback cells: {fb}{'  <-- INVESTIGATE' if fb else ''}", flush=True)
    return out

def build_training_ds(df):
    val_idx=int(df.loc[df["utc"]>=VAL_START,"time_idx"].min())
    return TimeSeriesDataSet(df[df["time_idx"]<val_idx],
        time_idx="time_idx",target="demand",group_ids=["zone"],
        max_encoder_length=ENCODER_LEN,max_prediction_length=DECODER_LEN,
        time_varying_unknown_reals=UNKNOWN_REALS,time_varying_known_reals=KNOWN_REALS,
        time_varying_known_categoricals=KNOWN_CATS,static_categoricals=["zone"],
        target_normalizer=GroupNormalizer(groups=["zone"]),
        add_relative_time_idx=True,add_target_scales=True,allow_missing_timesteps=False)

@torch.no_grad()
def predict_one_zone(model, training_ds, df_zone, tsi, device, debug=False):
    ds = TimeSeriesDataSet.from_dataset(training_ds, df_zone,
        min_prediction_idx=tsi, stop_randomization=True)
    dl = ds.to_dataloader(train=False, batch_size=BATCH, num_workers=0)
    yhat_parts, ytrue_parts, tidx_parts = [], [], []
    for bi, (x, (y, _)) in enumerate(dl):
        xd = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in x.items()}
        raw = model(xd)
        yhat = raw["prediction"][..., raw["prediction"].shape[-1] // 2].cpu().numpy()
        if debug and bi == 0:
            print(f"      [debug] pred_MW~{yhat[0,0]:.0f} true~{float(y[0,0]):.0f}", flush=True)
        yhat_parts.append(yhat); ytrue_parts.append(y.cpu().numpy())
        tidx_parts.append(x["decoder_time_idx"][:, 0].cpu().numpy())
        del xd, raw, yhat
    del dl, ds; gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
    return (np.concatenate(yhat_parts), np.concatenate(ytrue_parts), np.concatenate(tidx_parts))

pl.seed_everything(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
baseline = {int(k):v for k,v in json.load(open(BASE_JSON))["day_mape"].items()} if os.path.exists(BASE_JSON) else None

df=load_base()
leads=pd.read_parquet(LEADS); leads.index=pd.to_datetime(leads.index)
if getattr(leads.index,"tz",None) is not None: leads.index=leads.index.tz_localize(None)

tsi=int(df.loc[df["utc"]>=TEST_START,"time_idx"].min())
training_ds=build_training_ds(df)
model=TemporalFusionTransformer.load_from_checkpoint(CKPT).to(device).eval()

one=df[df["zone"]==df["zone"].iloc[0]]
month_arr=np.full(int(df["time_idx"].max())+DECODER_LEN+2,-1)
month_arr[one["time_idx"].values]=one["utc"].dt.month.values

df=df[df["time_idx"]>=tsi-ENCODER_LEN-1].copy()
print(f"trimmed to {len(df):,} rows, test_start_idx={tsi}, device={device}", flush=True)
ZONES=sorted(df["zone"].unique())

day_mape, composite = {}, []
for d in range(1,6):
    print(f"\n=== Lead {d} ===", flush=True)
    if os.path.exists(lead_file(d)):
        z=np.load(lead_file(d)); ape,months=z["ape"],z["months"]; print("  cached",flush=True)
    else:
        df_d=swap_lead(df,leads,d)
        hat_by_t,true_by_t={},{}
        for zi,zone in enumerate(ZONES):
            yh,yt,tt=predict_one_zone(model,training_ds,df_d[df_d["zone"]==zone],tsi,device,
                                      debug=(d==1 and zi==0))
            for i,t in enumerate(tt):
                hat_by_t.setdefault(int(t),[]).append(yh[i])
                true_by_t.setdefault(int(t),[]).append(yt[i])
            print(f"    {zone:10s} ({zi+1}/11)", flush=True)
        del df_d; gc.collect()
        good=sorted(t for t,v in hat_by_t.items() if len(v)==11)
        drop=len(hat_by_t)-len(good)
        if drop: print(f"  dropped {drop} incomplete windows",flush=True)
        sh=np.stack([np.sum(hat_by_t[t],axis=0) for t in good])
        st=np.stack([np.sum(true_by_t[t],axis=0) for t in good])
        ut=np.array(good); s,e=(d-1)*24,d*24
        ape=np.abs(st[:,s:e]-sh[:,s:e])/np.clip(st[:,s:e],1e-6,None)
        months=month_arr[ut[:,None]+np.arange(s,e)[None,:]]
        np.savez(lead_file(d),ape=ape,months=months)
        del hat_by_t,true_by_t; gc.collect()
    day_mape[d]=float(ape.mean()*100)
    b=f" (floor {[2.70,3.05,3.32,3.66,4.11][d-1]:.2f}%)"
    print(f"Day {d}: {day_mape[d]:.2f}%{b}  [saved]",flush=True)
    composite.append((ape.ravel(),months.ravel()))

all_ape=np.concatenate([a for a,_ in composite]); all_mon=np.concatenate([m for _,m in composite])
overall=float(all_ape.mean()*100)
smap={12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",6:"Summer",7:"Summer",8:"Summer",9:"Fall",10:"Fall",11:"Fall"}
seasons={sn:(float(all_ape[np.isin(all_mon,[m for m,x in smap.items() if x==sn])].mean()*100)
    if np.isin(all_mon,[m for m,x in smap.items() if x==sn]).any() else None)
    for sn in ["Winter","Spring","Summer","Fall"]}

FLOOR=[2.70,3.05,3.32,3.66,4.11]
print("\n===== ZONAL TUNED (lr=3e-4) LEAD-MATCHED RESULTS =====")
for d in range(1,6):
    print(f"Day {d}: {day_mape[d]:.2f}%   (floor {FLOOR[d-1]:.2f}%, statewide {baseline[d]:.2f}%)" if baseline
          else f"Day {d}: {day_mape[d]:.2f}%   (floor {FLOOR[d-1]:.2f}%)")
print(f"Overall: {overall:.2f}%   (floor 3.37%, statewide baseline 3.96%)")
print("\nSeasonal (floor: Winter 3.71, Spring 3.50, Summer 3.76, Fall 2.37):")
for sn,v in seasons.items(): print(f"  {sn}: {v:.2f}%" if v else f"  {sn}: n/a")

json.dump({"day_mape":{str(k):round(v,3) for k,v in day_mape.items()},
    "overall_mape":round(overall,3),
    "seasonal_mape":{k:(round(v,3) if v else None) for k,v in seasons.items()},
    "checkpoint":"zonal_tft_lr3e4_best.ckpt (epoch 1, lr=3e-4, val_loss 20.86)"},
    open(OUT_JSON,"w"),indent=2)
print(f"\nSaved -> {OUT_JSON}")

Seed set to 42
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


trimmed to 228,767 rows, test_start_idx=75080, device=cuda

=== Lead 1 ===
  fallback cells: 0
      [debug] pred_MW~1486 true~1518
    CAPITL     (1/11)
    CENTRL     (2/11)
    DUNWOD     (3/11)
    GENESE     (4/11)
    HUD VL     (5/11)
    LONGIL     (6/11)
    MHK VL     (7/11)
    MILLWD     (8/11)
    N.Y.C.     (9/11)
    NORTH      (10/11)
    WEST       (11/11)
Day 1: 2.48% (floor 2.70%)  [saved]

=== Lead 2 ===
  fallback cells: 0
    CAPITL     (1/11)
    CENTRL     (2/11)
    DUNWOD     (3/11)
    GENESE     (4/11)
    HUD VL     (5/11)
    LONGIL     (6/11)
    MHK VL     (7/11)
    MILLWD     (8/11)
    N.Y.C.     (9/11)
    NORTH      (10/11)
    WEST       (11/11)
Day 2: 2.89% (floor 3.05%)  [saved]

=== Lead 3 ===
  fallback cells: 0
    CAPITL     (1/11)
    CENTRL     (2/11)
    DUNWOD     (3/11)
    GENESE     (4/11)
    HUD VL     (5/11)
    LONGIL     (6/11)
    MHK VL     (7/11)
    MILLWD     (8/11)
    N.Y.C.     (9/11)
    NORTH      (10/11)
    WEST       

In [3]:
import subprocess, sys, os, getpass, importlib.util
if importlib.util.find_spec("mlflow") is None:
    subprocess.run([sys.executable,"-m","pip","install","-q","mlflow"])
    print("mlflow installed")

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/Sangi2805/Forecasting-Energy-Demand.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "Sangi2805"
os.environ["MLFLOW_TRACKING_PASSWORD"] = getpass.getpass("DagsHub token: ")

import json, mlflow
ROOT = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
OUT_JSON = f"{ROOT}/metrics_zonal_lr3e4.json"
m = json.load(open(OUT_JSON))

mlflow.set_experiment("Default")
with mlflow.start_run(run_name="tft_Sangar_zonal_leadmatched_lr3e4"):
    mlflow.log_params({
        "model": "TFT", "who": "Sangar", "data": "zonal_11zone_aggregated",
        "checkpoint": "zonal_tft_lr3e4_best.ckpt",
        "learning_rate": 3e-4, "reduce_on_plateau_patience": 2,
        "early_stop_patience": 6, "hidden_size": 64,
        "encoder_len": 168, "decoder_len": 120,
        "test_start_utc": "2024-01-23 12:00", "protocol": "5pass_lead_matched",
        "best_epoch": 1, "note": "tuned lr, converges 2-3 epochs then overfits",
    })
    mlflow.log_metric("overall_mape", m["overall_mape"])
    for d, v in m["day_mape"].items():
        mlflow.log_metric(f"day{d}_mape", v)
    for s, v in (m.get("seasonal_mape") or {}).items():
        if v is not None: mlflow.log_metric(f"{s.lower()}_mape", v)
    mlflow.log_metric("val_loss_best", 20.8578)
    mlflow.log_metric("floor_mape", 3.37)
    mlflow.log_metric("statewide_baseline_mape", 3.96)
    mlflow.log_artifact(OUT_JSON)
    print("logged run:", mlflow.active_run().info.run_id)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
feast 0.53.0 requires dill~=0.3.0, but you have dill 0.4.1 which is incompatible.
feast 0.53.0 requires pyarrow<=17.0.0, but you have pyarrow 24.0.0 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


mlflow installed


DagsHub token:  ········


logged run: 2d50e067e2634939967b5e0b866e9fea
🏃 View run tft_Sangar_zonal_leadmatched_lr3e4 at: https://dagshub.com/Sangi2805/Forecasting-Energy-Demand.mlflow/#/experiments/0/runs/2d50e067e2634939967b5e0b866e9fea
🧪 View experiment at: https://dagshub.com/Sangi2805/Forecasting-Energy-Demand.mlflow/#/experiments/0


In [4]:
import subprocess, os
os.chdir("/opt/app-root/src/Forecasting-Energy-Demand")
print(subprocess.run(["git","status","-sb"], capture_output=True, text=True).stdout)
print("--- remotes ---")
print(subprocess.run(["git","remote","-v"], capture_output=True, text=True).stdout)

## main...origin/main [ahead 4]
 M Sangar/01_download_nyiso_zonal.ipynb
 M Sangar/eval_lead_matched.ipynb
 M Sangar/nyiso_zonal_hourly.parquet
 M Shahriar/tft_training_2.ipynb
 M reports/mean_demand_by_humidity_range.png
 M reports/mean_demand_by_temperature_range.png
 M reports/plots/lstm_actual_vs_forecast.png
 M reports/plots/lstm_training_loss.png
 M requirements.txt
 M src/config.py
 M src/data_collection.py
 M src/eda.py
 M src/preprocessing.py
 M src/test.py
 M src/train_enhanced_lstm.py
 M src/train_lstm.py
?? Sangar/.~01_download_nyiso_zonal.ipynb
?? Sangar/.~02_download_zonal_weather.ipynb
?? Sangar/.~03_zonal_features.ipynb
?? Sangar/.~eval_lead_matched.ipynb
?? Sangar/01_feature_engineering.ipynb
?? Sangar/02_download_zonal_weather.ipynb
?? Sangar/02_tft_train.ipynb
?? Sangar/02_tft_train.py
?? Sangar/03_tft_evaluate.py
?? Sangar/03_zonal_features.ipynb
?? Sangar/04_zonal_tft_train.ipynb
?? Sangar/04_zonal_tft_train.ipynb.invalid
?? Sangar/04_zonal_tft_train.py
?? Sangar/05

In [5]:
import subprocess, os
os.chdir("/opt/app-root/src/Forecasting-Energy-Demand")

files = ["Sangar/04_zonal_tft_train.py",
         "Sangar/05_zonal_eval.py",
         "Sangar/metrics_zonal_lr3e4.json"]
print(subprocess.run(["git","add"]+files, capture_output=True, text=True).stderr or "staged")

print(subprocess.run(["git","status","-s","--"]+files, capture_output=True, text=True).stdout)

r = subprocess.run(["git","commit","-m",
     "Zonal TFT tuned (lr=3e-4): 3.20% lead-matched MAPE vs 3.37% floor, 3.96% statewide"],
     capture_output=True, text=True)
print(r.stdout or r.stderr)

staged
A  Sangar/04_zonal_tft_train.py
A  Sangar/05_zonal_eval.py
A  Sangar/metrics_zonal_lr3e4.json

[main 830bade] Zonal TFT tuned (lr=3e-4): 3.20% lead-matched MAPE vs 3.37% floor, 3.96% statewide
 3 files changed, 339 insertions(+)
 create mode 100644 Sangar/04_zonal_tft_train.py
 create mode 100644 Sangar/05_zonal_eval.py
 create mode 100644 Sangar/metrics_zonal_lr3e4.json



In [6]:
import subprocess, os
os.chdir("/opt/app-root/src/Forecasting-Energy-Demand")
r = subprocess.run(["git","push","sangar","main"], capture_output=True, text=True)
print(r.stdout, r.stderr)

 To https://github.com/Sangi2805/Forecasting-Energy-Demand.git
   b090447..830bade  main -> main



In [7]:
import subprocess, os
root = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
print("--- processed parquets ---")
for f in ["nyiso_zonal_hourly.parquet","weather_observed_zonal.parquet",
          "weather_leads_zonal.parquet","zonal_features.parquet"]:
    p = os.path.join(root,f)
    print(f"  {os.path.getsize(p)/1e6:8.1f} MB  {f}" if os.path.exists(p) else f"  MISSING   {f}")
print("--- raw dirs ---")
print(subprocess.run(["du","-sh", f"{root}/nyiso_zonal_raw", f"{root}/weather_cache", f"{root}/data"],
                     capture_output=True, text=True).stdout)
print("--- pipeline notebooks ---")
print(subprocess.run(["ls","-l"]+[f"{root}/{n}" for n in
     ["01_download_nyiso_zonal.ipynb","02_download_zonal_weather.ipynb","03_zonal_features.ipynb"]],
     capture_output=True, text=True).stdout)

--- processed parquets ---
       8.8 MB  nyiso_zonal_hourly.parquet
      13.4 MB  weather_observed_zonal.parquet
      11.9 MB  weather_leads_zonal.parquet
      66.8 MB  zonal_features.parquet
--- raw dirs ---
11M	/opt/app-root/src/Forecasting-Energy-Demand/Sangar/nyiso_zonal_raw
59M	/opt/app-root/src/Forecasting-Energy-Demand/Sangar/weather_cache
82M	/opt/app-root/src/Forecasting-Energy-Demand/Sangar/data

--- pipeline notebooks ---
-rw-rw-r--. 1 1000950000 1000950000 0 Jul 15 16:31 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/01_download_nyiso_zonal.ipynb
-rw-rw-r--. 1 1000950000 1000950000 0 Jul 15 16:31 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/02_download_zonal_weather.ipynb
-rw-rw-r--. 1 1000950000 1000950000 0 Jul 15 16:31 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/03_zonal_features.ipynb



In [8]:
import subprocess, os, glob
root = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
os.chdir("/opt/app-root/src/Forecasting-Energy-Demand")

print("--- git's copy of 01 ---")
r = subprocess.run(["git","cat-file","-s","HEAD:Sangar/01_download_nyiso_zonal.ipynb"],
                   capture_output=True, text=True)
print("size in HEAD:", r.stdout.strip() or r.stderr.strip())

print("\n--- autosave / invalid / checkpoint copies ---")
for p in sorted(glob.glob(f"{root}/.~*") + glob.glob(f"{root}/*.invalid")
                + glob.glob(f"{root}/.ipynb_checkpoints/*")):
    print(f"  {os.path.getsize(p):>9,} B  {os.path.basename(p)}")

--- git's copy of 01 ---
size in HEAD: 18012

--- autosave / invalid / checkpoint copies ---
         72 B  01_download_nyiso_zonal-checkpoint.ipynb
         72 B  01_feature_engineering-checkpoint.ipynb
         72 B  02_download_zonal_weather-checkpoint.ipynb
         72 B  02_tft_train-checkpoint.ipynb
      4,377 B  02_tft_train-checkpoint.py
      3,915 B  03_tft_evaluate-checkpoint.py
         72 B  03_zonal_features-checkpoint.ipynb
     28,472 B  04_zonal_tft_train-checkpoint.ipynb
         72 B  05_zonal_eval-checkpoint.ipynb
     20,166 B  05_zonal_eval_tuned-checkpoint.ipynb
         72 B  Evaluate-checkpoint.ipynb
         72 B  eval_lead_matched-checkpoint.ipynb
         72 B  tft_train_clean-checkpoint.ipynb
     31,362 B  .~01_download_nyiso_zonal.ipynb
     12,205 B  .~02_download_zonal_weather.ipynb
     12,553 B  .~03_zonal_features.ipynb
     27,181 B  .~eval_lead_matched.ipynb
          0 B  04_zonal_tft_train.ipynb.invalid
          0 B  05_zonal_eval.ipynb.invalid

In [9]:
import json, os, shutil
root = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"

pairs = [("01_download_nyiso_zonal.ipynb", ".~01_download_nyiso_zonal.ipynb"),
         ("02_download_zonal_weather.ipynb", ".~02_download_zonal_weather.ipynb"),
         ("03_zonal_features.ipynb", ".~03_zonal_features.ipynb")]

for target, source in pairs:
    src = os.path.join(root, source)
    dst = os.path.join(root, target)
    try:
        nb = json.load(open(src))
        ncells = len(nb.get("cells", []))
        ncode = sum(1 for c in nb["cells"] if c["cell_type"] == "code")
        shutil.copy2(dst, dst + ".empty_backup")
        shutil.copy2(src, dst)
        print(f"RESTORED {target}: {ncells} cells ({ncode} code), {os.path.getsize(dst):,} B")
    except Exception as e:
        print(f"FAILED   {target}: {type(e).__name__}: {e}")

RESTORED 01_download_nyiso_zonal.ipynb: 7 cells (7 code), 31,362 B
RESTORED 02_download_zonal_weather.ipynb: 2 cells (2 code), 12,205 B
RESTORED 03_zonal_features.ipynb: 5 cells (5 code), 12,553 B


In [10]:
import subprocess, os
os.chdir("/opt/app-root/src/Forecasting-Energy-Demand")

files = ["Sangar/01_download_nyiso_zonal.ipynb",
         "Sangar/02_download_zonal_weather.ipynb",
         "Sangar/03_zonal_features.ipynb"]
subprocess.run(["git","add"]+files)
print(subprocess.run(["git","status","-s","--"]+files, capture_output=True, text=True).stdout)

r = subprocess.run(["git","commit","-m",
     "Zonal data pipeline: NYISO download, weather download, feature engineering"],
     capture_output=True, text=True)
print(r.stdout or r.stderr)

r = subprocess.run(["git","push","sangar","main"], capture_output=True, text=True)
print(r.stdout, r.stderr)

M  Sangar/01_download_nyiso_zonal.ipynb
A  Sangar/02_download_zonal_weather.ipynb
A  Sangar/03_zonal_features.ipynb

[main 7b6ae28] Zonal data pipeline: NYISO download, weather download, feature engineering
 3 files changed, 1035 insertions(+), 1 deletion(-)
 create mode 100644 Sangar/02_download_zonal_weather.ipynb
 create mode 100644 Sangar/03_zonal_features.ipynb

 To https://github.com/Sangi2805/Forecasting-Energy-Demand.git
   830bade..7b6ae28  main -> main



In [11]:
import json, re
nb = json.load(open("/opt/app-root/src/Forecasting-Energy-Demand/Sangar/01_download_nyiso_zonal.ipynb"))
src = "\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"]=="code")
for m in sorted(set(re.findall(r'https?://[^\s"\')]+', src))):
    print(m)

http://mis.nyiso.com/public/csv/palIntegrated
https://previous-runs-api.open-meteo.com/v1/forecast


In [12]:
import os, gc
import numpy as np, pandas as pd, torch
import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

FOLDER   = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
FEATURES = os.path.join(FOLDER, "zonal_features.parquet")
CKPT     = os.path.join(FOLDER, "checkpoints_zonal", "zonal_tft_lr3e4_best.ckpt")

TRAIN_START="2015-07-01 04:00"; VAL_START="2023-01-01 05:00"; TEST_START="2024-01-23 12:00"
ENCODER_LEN, DECODER_LEN = 168, 120
BATCH, SEED = 128, 42
TARGET_BATCHES = 25          # sampled batches per zone per split

LEAN_VARS=["temperature_2m","apparent_temperature","relative_humidity_2m",
           "wind_speed_10m","shortwave_radiation","cloud_cover"]
UNKNOWN_REALS=["demand","demand_lag24","demand_lag168",
               "demand_roll24_mean","demand_roll168_mean","demand_roll24_std"]
KNOWN_REALS=LEAN_VARS+["temp_vshape","time_idx"]
KNOWN_CATS=["hour","day_of_week","month","is_weekend","is_holiday"]

def load_base():
    df=pd.read_parquet(FEATURES); df["utc"]=pd.to_datetime(df["utc"])
    for c in KNOWN_CATS: df[c]=df[c].astype(str).astype("category")
    df["zone"]=df["zone"].astype(str)
    df=df[df["utc"]>=TRAIN_START].copy()
    df["time_idx"]=df["time_idx"]-df["time_idx"].min()
    return df.sort_values(["zone","time_idx"]).reset_index(drop=True)

def build_training_ds(df, val_idx):
    return TimeSeriesDataSet(df[df["time_idx"]<val_idx],
        time_idx="time_idx",target="demand",group_ids=["zone"],
        max_encoder_length=ENCODER_LEN,max_prediction_length=DECODER_LEN,
        time_varying_unknown_reals=UNKNOWN_REALS,time_varying_known_reals=KNOWN_REALS,
        time_varying_known_categoricals=KNOWN_CATS,static_categoricals=["zone"],
        target_normalizer=GroupNormalizer(groups=["zone"]),
        add_relative_time_idx=True,add_target_scales=True,allow_missing_timesteps=False)

@torch.no_grad()
def predict_zone(model, training_ds, df_zone, start_idx, device, batch_ids):
    ds = TimeSeriesDataSet.from_dataset(training_ds, df_zone,
        min_prediction_idx=start_idx, stop_randomization=True)
    dl = ds.to_dataloader(train=False, batch_size=BATCH, num_workers=0)
    n = len(dl)
    if batch_ids is None:
        stride = max(1, n // TARGET_BATCHES)
        batch_ids = set(range(0, n, stride))
    yh, yt, tt = [], [], []
    for bi, (x, (y, _)) in enumerate(dl):
        if bi not in batch_ids: continue
        xd = {k:(v.to(device) if torch.is_tensor(v) else v) for k,v in x.items()}
        raw = model(xd)
        yh.append(raw["prediction"][..., raw["prediction"].shape[-1]//2].cpu().numpy())
        yt.append(y.cpu().numpy())
        tt.append(x["decoder_time_idx"][:,0].cpu().numpy())
        del xd, raw
    del dl, ds; gc.collect()
    if device=="cuda": torch.cuda.empty_cache()
    return np.concatenate(yh), np.concatenate(yt), np.concatenate(tt), batch_ids

def eval_split(name, model, training_ds, df, start_idx, end_idx, zones, device):
    sub = df[(df["time_idx"]>=start_idx-ENCODER_LEN-1)&(df["time_idx"]<=end_idx)]
    hat, tru, batch_ids = {}, {}, None
    for zone in zones:
        yh, yt, tt, batch_ids = predict_zone(
            model, training_ds, sub[sub["zone"]==zone], start_idx, device, batch_ids)
        for i,t in enumerate(tt):
            hat.setdefault(int(t),[]).append(yh[i])
            tru.setdefault(int(t),[]).append(yt[i])
    good = sorted(t for t,v in hat.items() if len(v)==11)
    sh = np.stack([np.sum(hat[t],axis=0) for t in good])
    st = np.stack([np.sum(tru[t],axis=0) for t in good])
    ape = np.abs(st-sh)/np.clip(st,1e-6,None)
    overall = float(ape.mean()*100)
    per_day = [float(ape[:,(d-1)*24:d*24].mean()*100) for d in range(1,6)]
    print(f"{name:5s}  windows={len(good):5d}  MAPE={overall:5.2f}%   "
          f"day1-5: {' '.join(f'{v:.2f}' for v in per_day)}", flush=True)
    return overall

pl.seed_everything(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
df = load_base()
val_idx  = int(df.loc[df["utc"]>=VAL_START ,"time_idx"].min())
test_idx = int(df.loc[df["utc"]>=TEST_START,"time_idx"].min())
max_idx  = int(df["time_idx"].max())
training_ds = build_training_ds(df, val_idx)
model = TemporalFusionTransformer.load_from_checkpoint(CKPT).to(device).eval()
zones = sorted(df["zone"].unique())
print(f"boundaries  train:[{ENCODER_LEN+1},{val_idx-1}]  "
      f"val:[{val_idx},{test_idx-1}]  test:[{test_idx},{max_idx}]\n", flush=True)

r = {}
r["TRAIN"] = eval_split("TRAIN", model, training_ds, df, ENCODER_LEN+1, val_idx-1, zones, device)
r["VAL"]   = eval_split("VAL",   model, training_ds, df, val_idx,      test_idx-1, zones, device)
r["TEST"]  = eval_split("TEST",  model, training_ds, df, test_idx,     max_idx,    zones, device)

print("\n================ DIAGNOSIS ================")
print(f"train {r['TRAIN']:.2f}%   val {r['VAL']:.2f}%   test {r['TEST']:.2f}%   (all observed weather)")
print(f"test with real forecasts (lead-matched): 3.20%")
print(f"  train->val  gap: {r['VAL']-r['TRAIN']:+.2f} pts")
print(f"  val->test   gap: {r['TEST']-r['VAL']:+.2f} pts")
print(f"  weather-forecast penalty: {3.20-r['TEST']:+.2f} pts")

Seed set to 42


boundaries  train:[169,65784]  val:[65785,75079]  test:[75080,95707]



/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


TRAIN  windows= 3328  MAPE= 0.97%   day1-5: 0.99 0.97 0.97 0.94 0.96
VAL    windows= 4608  MAPE= 1.39%   day1-5: 1.30 1.40 1.42 1.42 1.41
TEST   windows= 3456  MAPE= 2.25%   day1-5: 2.14 2.21 2.27 2.30 2.35

================ DIAGNOSIS ================
train 0.97%   val 1.39%   test 2.25%   (all observed weather)
test with real forecasts (lead-matched): 3.20%
  train->val  gap: +0.42 pts
  val->test   gap: +0.87 pts
  weather-forecast penalty: +0.95 pts
